# TRANSFORMAÇÃO DE COORDENADAS #


Estes scripts foram adaptados do website do INPE: http://www.dpi.inpe.br/calcula/

In [ ]:
import math
import pandas as pd

In [ ]:
def read_files(filepath, output_path=None):
    file = pd.read_csv(filepath)
    results = []
    if output_path:
        out_df = pd.DataFrame(results, columns=['lat2', 'lon2', 'h2'])
        out_df.to_csv(output_path, index=False)
    return results

In [ ]:
# Datum parameters (id, name, semi_axis, flattening, deltax, deltay, deltaz)
datum_parameters = [
    ("0", "Null", 0.0, 0.0, 0.0, 0.0, 0.0),
    ("1", "SAD69", 6378160.0, 0.00335289187, -67.35, 3.88, -38.22),
    ("2", "CorregoAlegre", 6378388.0, 0.00336700337, -206.05, 168.28, -3.82),
    ("3", "AstroChua", 6378388.0, 0.00336700337, -144.35, 243.37, -33.22),
    ("4", "WGS84", 6378137.0, 0.00335281066, 0.0, 0.0, 0.0),
    ("5", "SIRGAS2000", 6378137.0, 0.00335281068, 0.0, 0.0, 0.0),
]

In [ ]:
"""
 geo_to_utm				       
 Autor		: Julio Cesar Lima d'Alge		       jul-88
 Conversao p/ Php	: Luis Maurano						 2008
 Resumo		: transforma coordenadas geodesicas em coordenadas UTM
 Entradas	: 
 			lat		coordenadas geodesicas (em radianos)
		  lon		coordenadas geodesicas (em radianos)
		  lon_mc	meridiano central (em radianos)
		  semi_eixo	semi_eixo_maior do elipsoide
		  achat		achatamento do elipsoide
			hemis		norte ou sul
 Saidas		: x y coordenadas UTM (em metros)
"""

def geo_to_utm(lat, lon, semi_axis, flattening, hemisphere, lon_mc):
    pi = math.pi
    offy = 0.0 if hemisphere == "norte" else 10000000.0
    k0 = 1.0 - (1.0 / 2500.0)
    equad = 2.0 * flattening - flattening ** 2
    elinquad = equad / (1.0 - equad)
    aux1 = equad ** 2
    aux2 = aux1 * equad
    aux3 = math.sin(2.0 * lat)
    aux4 = math.sin(4.0 * lat)
    aux5 = math.sin(6.0 * lat)
    aux6 = (1.0 - equad / 4.0 - 3.0 * aux1 / 64.0 - 5.0 * aux2 / 256.0) * lat
    aux7 = (3.0 * equad / 8.0 + 3.0 * aux1 / 32.0 + 45.0 * aux2 / 1024.0) * aux3
    aux8 = (15.0 * aux1 / 256.0 + 45.0 * aux2 / 1024.0) * aux4
    aux9 = (35.0 * aux2 / 3072.0) * aux5
    n = semi_axis / math.sqrt(1.0 - equad * math.sin(lat) ** 2)
    t = math.tan(lat) ** 2
    c = elinquad * math.cos(lat) ** 2
    ag = (lon - lon_mc) * math.cos(lat)
    m = semi_axis * (aux6 - aux7 + aux8 - aux9)
    aux10 = (1.0 - t + c) * ag ** 3 / 6.0
    aux11 = (5.0 - 18.0 * t + t ** 2 + 72.0 * c - 58.0 * elinquad) * ag ** 5 / 120.0
    aux12 = (5.0 - t + 9.0 * c + 4.0 * c ** 2) * ag ** 4 / 24.0
    aux13 = (61.0 - 58.0 * t + t ** 2 + 600.0 * c - 330.0 * elinquad) * ag ** 6 / 720.0
    x = 500000.0 + k0 * n * (ag + aux10 + aux11)
    y = offy + k0 * (m + n * math.tan(lat) * (ag ** 2 / 2.0 + aux12 + aux13))
    return x, y

def utm_to_geo(x, y, semi_axis, flattening, hemisphere, lon_mc):
    pi = math.pi
    if hemisphere == "norte":
        y += 10000000.0
    k0 = 1.0 - (1.0 / 2500.0)
    equad = 2.0 * flattening - flattening ** 2
    elinquad = equad / (1.0 - equad)
    e1 = (1.0 - math.sqrt(1.0 - equad)) / (1.0 + math.sqrt(1.0 - equad))
    aux1 = equad ** 2
    aux2 = aux1 * equad
    aux3 = e1 ** 2
    aux4 = e1 * aux3
    aux5 = aux4 * e1
    m = (y - 10000000.0) / k0
    mi = m / (semi_axis * (1.0 - equad / 4.0 - 3.0 * aux1 / 64.0 - 5.0 * aux2 / 256.0))
    aux6 = (3.0 * e1 / 2.0 - 27.0 * aux4 / 32.0) * math.sin(2.0 * mi)
    aux7 = (21.0 * aux3 / 16.0 - 55.0 * aux5 / 32.0) * math.sin(4.0 * mi)
    aux8 = (151.0 * aux4 / 96.0) * math.sin(6.0 * mi)
    lat1 = mi + aux6 + aux7 + aux8
    c1 = elinquad * math.cos(lat1) ** 2
    t1 = math.tan(lat1) ** 2
    n1 = semi_axis / math.sqrt(1.0 - equad * math.sin(lat1) ** 2)
    quoc = (1.0 - equad * math.sin(lat1) ** 2) ** 3
    r1 = semi_axis * (1.0 - equad) / math.sqrt(quoc)
    d = (x - 500000.0) / (n1 * k0)
    aux9 = (5.0 + 3.0 * t1 + 10.0 * c1 - 4.0 * c1 ** 2 - 9.0 * elinquad) * d ** 4 / 24.0
    aux10 = (61.0 + 90.0 * t1 + 298.0 * c1 + 45.0 * t1 ** 2 - 252.0 * elinquad - 3.0 * c1 ** 2) * d ** 6 / 720.0
    aux11 = d - (1.0 + 2.0 * t1 + c1) * d ** 3 / 6.0
    aux12 = (5.0 - 2.0 * c1 + 28.0 * t1 - 3.0 * c1 ** 2 + 8.0 * elinquad + 24.0 * t1 ** 2) * d ** 5 / 120.0
    lat = lat1 - (n1 * math.tan(lat1) / r1) * (d ** 2 / 2.0 - aux9 + aux10)
    lon = lon_mc + (aux11 + aux12) / math.cos(lat1)
    return lat, lon

def define_mer_cent(lon):
    """ define_mer_cent			       	       
    Autor		: Julio Cesar Lima d'Alge		       fev-90
    Conversao p/ Php	: Luis Maurano						 2008
    Conversao p/ Python : Luiza Werli                        2025
    Resumo		: calcula meridianos centrais dos fusos UTM e GAUSS
    a partir da longitude geodesica
    Entradas	: lon (em grau decimal)
    Saidas		: mc1_utm,mc2_utm,mc1_gauss,mc2_gauss
    Obs.		: os dois valores para UTM e Gauss referem-se 'as
    situacoes de bordas de fusos
    """
    if lon != 0.0:
        sinal = int(lon / abs(lon))
    else:
        return 3.0, -3.0
    ind1 = 0
    ind2 = 6
    k = 0
    while abs(lon) > ind2:
        k += 1
        ind1 = 6 * k
        ind2 = ind1 + 6
    mc = ind1 + 3.0
    if abs(lon) < mc and lon != 0.0:
        mc1_utm = mc2_utm = mc
    elif abs(lon) > mc and abs(lon) != mc + 3.0:
        mc1_utm = mc2_utm = mc
    elif abs(lon) == mc + 3.0 and abs(lon) != 180.0:
        mc1_utm = mc
        mc2_utm = mc + 6.0
    elif lon == 180.0:
        mc1_utm = 177.0
        mc2_utm = -177.0
    elif lon == -180.0:
        mc1_utm = -177.0
        mc2_utm = 177.0
    elif abs(lon) == mc:
        mc1_utm = mc2_utm = mc
    else:
        mc1_utm = mc2_utm = mc
    if lon != 0.0 and abs(lon) != 180:
        mc1_utm *= sinal
        mc2_utm *= sinal
    return mc1_utm, mc2_utm


def datum1_to_datum2_from_file(filepath, semi_eixo, achat, deltax, deltay, deltaz,
    semi_eixo2, achat2, deltax2, deltay2, deltaz2, output_path=None):
    """
    Lê coordenadas geodésicas (lat, lon, h) de um arquivo CSV ou TXT e converte para o datum 2.
    O arquivo deve ter colunas: lat, lon, h (em radianos).
    Salva o resultado em um novo arquivo se output_path for fornecido.
    Retorna uma lista de tuplas (lat2, lon2, h2).
    """

    file = pd.read_csv(filepath)
    results = []

    for idx, row in file.iterrows():
        lon1 = row['LESTE']
        lat1 = row['NORTE']
        h1 = row['ALTITUDE']

        # calcula coordenadas geocentricas cartesianas no datum 1
        equad1 = 2.0 * achat - (achat ** 2)
        n1 = semi_eixo / math.sqrt(1.0 - equad1 * math.sin(lat1) ** 2)
        x1 = (n1 + h1) * math.cos(lat1) * math.cos(lon1)
        y1 = (n1 + h1) * math.cos(lat1) * math.sin(lon1)
        z1 = (n1 * (1.0 - equad1) + h1) * math.sin(lat1)

        # calcula coordenadas geocentricas cartesianas no datum 2
        x2 = x1 + (deltax - deltax2)
        y2 = y1 + (deltay - deltay2)
        z2 = z1 + (deltaz - deltaz2)

        # calcula coordenadas geodesicas no datum 2
        equad2 = 2.0 * achat2 - (achat2 ** 2)
        lat2 = lat1
        while True:
            n2 = semi_eixo2 / math.sqrt(1.0 - equad2 * math.sin(lat2) ** 2)
            lat2_new = math.atan((z2 + n2 * equad2 * math.sin(lat2)) / math.sqrt(x2 ** 2 + y2 ** 2))
            d = semi_eixo2 / math.sqrt(1.0 - equad2 * math.sin(lat2_new) ** 2) - n2
            if abs(d) <= 1e-11:
                lat2 = lat2_new
                break
            lat2 = lat2_new
        lon2 = math.atan2(y2, x2)
        h2 = h1

        results.append((lat2, lon2, h2))
        
# Salva em arquivo se output_path for fornecido
    if output_path:
        out_df = pd.DataFrame(results, columns=['lat2', 'lon2', 'h2'])
        out_df.to_csv(output_path, index=False)

    return results

# Exemplo de uso:
# results = datum1_to_datum2_from_file('coordenadas.csv', semi_eixo, achat, deltax, deltay, deltaz,
#                                      semi_eixo2, achat2, deltax2, deltay2, deltaz2, 'saida.csv')

def utm_datum_transform(x, y, h, semi_eixo1, achat1, deltax1, deltay1, deltaz1,
                       semi_eixo2, achat2, deltax2, deltay2, deltaz2,
                       hemisphere, lon_mc):
    """
    Converte coordenadas UTM (x, y, h) do datum 1 para UTM no datum 2.
    """

    # 1. UTM -> Geodésicas no datum 1
    lat1, lon1 = utm_to_geo(x, y, semi_eixo1, achat1, hemisphere, lon_mc)
    
    # 2. Geodésicas datum 1 -> datum 2
    lat2, lon2, h2 = datum1_to_datum2_from_file(
        semi_eixo1, achat1, deltax1, deltay1, deltaz1,
        lat1, lon1, h,
        semi_eixo2, achat2, deltax2, deltay2, deltaz2
    )
    
    # 3. Geodésicas datum 2 -> UTM
    x2, y2 = geo_to_utm(lat2, lon2, semi_eixo2, achat2, hemisphere, lon_mc)
    
    return x2, y2, h2

# Exemplo de uso:
# x, y = coordenadas UTM no datum 1
# h = altitude (se não tiver, use 0)
# semi_eixo1, achat1, deltax1, deltay1, deltaz1 = parâmetros do datum 1
# semi_eixo2, achat2, deltax2, deltay2, deltaz2 = parâmetros do datum 2
# hemisphere = "norte" ou "sul"
# lon_mc = meridiano central da zona UTM

# x2, y2, h2 = utm_datum_transform(x, y, h, semi_eixo1, achat1, deltax1, deltay1, deltaz1,
#                                  semi_eixo2, achat2, deltax2, deltay2, deltaz2,
#                                  hemisphere, lon_mc)

def geo_to_merc(lat, lon, lat1, longO, semi_eixo, achat, x, y):
    equad = 2.*achat - pow(achat,(double)2)
    aux1 = (1 + math.tan(lat/(double)2))/(1 - tan(lat/(double)2))
    aux2 = (equad+equad*equad/4.+equad*equad*equad/8.)*math.sin(lat)
    aux3 = (equad*equad/12.+equad*equad*equad/16.)*math.sin**3*lat
    aux4 = (equad*equad*equad/80.)*math.sin^5*lat
    aux5 = math.cos(lat1)
    aux6 = 1./math.sqrt((double)1-equad*pow(sin(lat1),(double)2))

    x = semi_eixo*(lon - longO)*aux5*aux6
    y = semi_eixo*(math.log(aux1) - aux2 + aux3 - aux4)*aux5*aux6

    return x, y

In [ ]:
entrada = r'E:\DOCUMENTOS SILVIO\COORDENADAS_MODIFICADO.csv'
saida = r'E:\DOCUMENTOS SILVIO\outputs\COORDENADAS_SIRGAS2000_VC.csv'

results = datum1_to_datum2_from_file(entrada, 6378160.0, 0.00335289187, -67.35, 3.88, -38.22, 
                                     6378137.0, 0.00335281068, 0.0, 0.0, 0.0, saida)